In [27]:
class SimpleReflexLightAgent:
    def __init__(self):
        pass
    
    def act(self, occupancy_status):
        actions = {}
        for room, occupied in occupancy_status.items():
            if occupied:
                actions[room] = "Turn ON"
            else:
                actions[room] = "Turn OFF"
        return actions

occupancy = {
    "Room-1" : False,
    "Room-2" : True,
    "Room-3" : False,
    "Room-4" : True,
    "Room-5" : False
}

agent = SimpleReflexLightAgent()

actions = agent.act(occupancy)

for room in occupancy:
    print(f"{room} : Occupied = {occupancy[room]}, Action = {actions[room]}")

Room-1 : Occupied = False, Action = Turn OFF
Room-2 : Occupied = True, Action = Turn ON
Room-3 : Occupied = False, Action = Turn OFF
Room-4 : Occupied = True, Action = Turn ON
Room-5 : Occupied = False, Action = Turn OFF


In [28]:
class ModelBasedWarehouseAgent:
    def __init__(self):
        self.model = {} 
        self.location = "A1"
        self.pick_list = ["item1", "item3"] 
        self.collected = []
    
    def update_model(self, percept):
        shelf, has_item, item_type = percept
        self.model[shelf] = (has_item, item_type)
    
    def act(self, percept):
        shelf, has_item, item_type = percept
        
        self.update_model((shelf, has_item, item_type))
        
        if has_item and item_type in self.pick_list and item_type not in self.collected:
            self.collected.append(item_type)
            return "Pick"
        
        if len(self.collected) == len(self.pick_list):
            return "Deliver"
        
        if self.location == "A1":
            self.location = "A2"
        elif self.location == "A2":
            self.location = "A3"
        else:
            self.location = "A1"
        
        return "Move"

environment = {
    "A1": (True, "item1"),
    "A2": (False, None),
    "A3": (True, "item3")
}

agent = ModelBasedWarehouseAgent()

for _ in range(6):
    shelf = agent.location
    has_item, item_type = environment[shelf]
    action = agent.act((shelf, has_item, item_type))
    
    print(f"Shelf: {shelf}, Item: {item_type}, Action: {action}")
    
    if action == "Pick":
        environment[shelf] = (False, None)

Shelf: A1, Item: item1, Action: Pick
Shelf: A1, Item: None, Action: Move
Shelf: A2, Item: None, Action: Move
Shelf: A3, Item: item3, Action: Pick
Shelf: A3, Item: None, Action: Deliver
Shelf: A3, Item: None, Action: Deliver


In [29]:
class ModelBasedAgent:
    def __init__(self):
        
        self.model = {
            "RoomA": "Unknown",
            "RoomB": "Unknown"
        }
        self.current_room = "RoomA"

    def update_model(self, percept):
        
        self.model[self.current_room] = percept

    def act(self, percept):
        self.update_model(percept)

        if percept == "Dirty":
            return "Suck"
        else:
            if self.current_room == "RoomA":
                self.current_room = "RoomB"
            else:
                self.current_room = "RoomA"
            return "Move"


environment = {
    "RoomA": "Dirty",
    "RoomB": "Clean"
}


agent = ModelBasedAgent()


for _ in range(4):
    room = agent.current_room
    percept = environment[room]
    action = agent.act(percept)

    print(f"Room: {room}, Percept: {percept}, Action: {action}")
   
    if action == "Suck":
        environment[room] = "Clean"

Room: RoomA, Percept: Dirty, Action: Suck
Room: RoomA, Percept: Clean, Action: Move
Room: RoomB, Percept: Clean, Action: Move
Room: RoomA, Percept: Clean, Action: Move


In [30]:
class GoalBasedAgent:
    def __init__(self, environment, goal):
        self.environment = environment  
        self.goal = goal                
        self.current_room = "RoomA"

    def is_goal_achieved(self):
        
        return all(self.environment[room] == self.goal[room] for room in self.environment)

    def act(self):
        
        percept = self.environment[self.current_room]

        if percept == "Dirty":
            
            self.environment[self.current_room] = "Clean"
            return "Suck"
        else:
            
            for room in self.environment:
                if self.environment[room] == "Dirty":
                    self.current_room = room
                    return f"Move to {room}"
            
            return "No Action, Goal Achieved"

environment = {"RoomA": "Dirty", "RoomB": "Dirty"}
goal = {"RoomA": "Clean", "RoomB": "Clean"}

agent = GoalBasedAgent(environment, goal)

while not agent.is_goal_achieved():
    action = agent.act()
    print(f"Current Room: {agent.current_room}, Action: {action}")

print("Goal Achieved! All rooms are clean")

Current Room: RoomA, Action: Suck
Current Room: RoomB, Action: Move to RoomB
Current Room: RoomB, Action: Suck
Goal Achieved! All rooms are clean


In [31]:
class UtilityBasedAgent:
    def __init__(self, environment):
        self.environment = environment
        self.current_room = "RoomA"
        self.moves = 0 

    def utility(self):
        clean_rooms = sum(1 for room in self.environment if self.environment[room] == "Clean")
        return clean_rooms - self.moves

    def act(self):
        percept = self.environment[self.current_room]

        if percept == "Dirty":
            self.environment[self.current_room] = "Clean"
            return "Suck"
        else:
            
            dirty_rooms = [room for room in self.environment if self.environment[room] == "Dirty"]
            if dirty_rooms:
                
                self.current_room = dirty_rooms[0]
                self.moves += 1
                return f"Move to {self.current_room}"
            else:
                return "No Action, All Clean"

environment = {"RoomA": "Dirty", "RoomB": "Dirty", "RoomC": "Clean"}
agent = UtilityBasedAgent(environment)

while any(environment[room] == "Dirty" for room in environment):
    action = agent.act()
    print(f"Current Room: {agent.current_room}, Action: {action}, Utility: {agent.utility()}")

print("All rooms clean! Final Utility:", agent.utility())

Current Room: RoomA, Action: Suck, Utility: 2
Current Room: RoomB, Action: Move to RoomB, Utility: 1
Current Room: RoomB, Action: Suck, Utility: 2
All rooms clean! Final Utility: 2


In [32]:
import random

class LearningBasedAgent:
    def __init__(self, actions):
        self.Q = {}
        self.actions = actions
        self.alpha = 0.1 
        self.gamma = 0.9 
        self.epsilon = 0.1 
    
    def get_Q_value(self, state, action):
        return self.Q.get((state, action), 0.0)
    
    def select_action(self, state):
        if random.uniform(0, 1) < self.epsilon:
            return random.choice(self.actions)
        else:
            return max(self.actions, key=lambda a: self.get_Q_value(state, a))
    
    def learn(self, state, action, reward, next_state):
        old_Q = self.get_Q_value(state, action)
        best_future_Q = max([self.get_Q_value(next_state, a) for a in self.actions])
        
        self.Q[(state, action)] = old_Q + self.alpha * (reward + self.gamma * best_future_Q - old_Q)
    
    def act(self, state):
        action = self.select_action(state)
        return action


class Environment:
    def __init__(self, state='Dirty'):
        self.state = state
    
    def get_percept(self):
        return self.state
    
    def clean_room(self):
        self.state = 'Clean'
        return 10
    
    def no_action_reward(self):
        return 0


def run_agent(agent, environment, steps):
    for step in range(steps):
        percept = environment.get_percept()
        action = agent.act(percept)
        
        if percept == 'Dirty':
            reward = environment.clean_room()
            print(f"Step {step + 1}: Percept - {percept}, Action - {action}, Reward - {reward}")
        else:
            reward = environment.no_action_reward()
            print(f"Step {step + 1}: Percept - {percept}, Action - {action}, Reward - {reward}")
        
        next_percept = environment.get_percept()
        agent.learn(percept, action, reward, next_percept)

agent = LearningBasedAgent(['Clean the room', 'No action needed'])
environment = Environment()

run_agent(agent, environment, 5)

Step 1: Percept - Dirty, Action - Clean the room, Reward - 10
Step 2: Percept - Clean, Action - Clean the room, Reward - 0
Step 3: Percept - Clean, Action - Clean the room, Reward - 0
Step 4: Percept - Clean, Action - Clean the room, Reward - 0
Step 5: Percept - Clean, Action - Clean the room, Reward - 0


In [33]:
graph = {
    "Control Room": ["Storage A", "Storage B"],
    "Storage A": ["Lab 1", "Lab 2"],
    "Storage B": ["Server Room"],
    "Lab 1": ["Debris 1"],
    "Lab 2": ["Debris 2"],
    "Debris 1": ["Maintenance Corridor"],
    "Debris 2": ["Maintenance Corridor"],
    "Server Room": ["Charging Bay"],
    "Charging Bay": ["Maintenance Corridor"],
    "Maintenance Corridor": ["Survivor Room"],
    "Survivor Room": []
}

def bfs(graph, start, goal):
    visited = set()
    queue = [start]
    visited.add(start)
    parent = {start: None}
    expansion_order = []

    while queue:
        node = queue.pop(0)
        expansion_order.append(node)

        if node == goal:
            path = []
            curr = node
            while curr is not None:
                path.append(curr)
                curr = parent[curr]
            path.reverse()
            return expansion_order, path

        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                visited.add(neighbor)
                parent[neighbor] = node
                queue.append(neighbor)

    return expansion_order, None

start_node = "Control Room"
goal_node = "Survivor Room"
order, path = bfs(graph, start_node, goal_node)

print("BFS Order of node expansion:", order)
print("BFS Path to goal:", path)

BFS Order of node expansion: ['Control Room', 'Storage A', 'Storage B', 'Lab 1', 'Lab 2', 'Server Room', 'Debris 1', 'Debris 2', 'Charging Bay', 'Maintenance Corridor', 'Survivor Room']
BFS Path to goal: ['Control Room', 'Storage A', 'Lab 1', 'Debris 1', 'Maintenance Corridor', 'Survivor Room']


In [34]:
graph = {
    "Control Room": ["Storage A", "Storage B"],
    "Storage A": ["Lab 1", "Lab 2"],
    "Storage B": ["Server Room"],
    "Lab 1": ["Debris 1"],
    "Lab 2": ["Debris 2"],
    "Debris 1": ["Maintenance Corridor"],
    "Debris 2": ["Maintenance Corridor"],
    "Server Room": ["Charging Bay"],
    "Charging Bay": ["Maintenance Corridor"],
    "Maintenance Corridor": ["Survivor Room"],
    "Survivor Room": []
}

def dfs_recursive(node, goal, visited, parent, expansion):
    visited.add(node)
    expansion.append(node)

    if node == goal:
        return True

    for neighbor in graph.get(node, []):
        if neighbor not in visited:
            parent[neighbor] = node
            if dfs_recursive(neighbor, goal, visited, parent, expansion):
                return True
    return False

start_node = "Control Room"
goal_node = "Survivor Room"
visited_set = set()
parent_dict = {start_node: None}
expansion_order = []

found = dfs_recursive(start_node, goal_node, visited_set, parent_dict, expansion_order)

if found:
    path = []
    curr = goal_node
    while curr is not None:
        path.append(curr)
        curr = parent_dict[curr]
    path.reverse()
else:
    path = None

print("DFS Recursive Order of node expansion:", expansion_order)
print("DFS Recursive Path to goal:", path)

DFS Recursive Order of node expansion: ['Control Room', 'Storage A', 'Lab 1', 'Debris 1', 'Maintenance Corridor', 'Survivor Room']
DFS Recursive Path to goal: ['Control Room', 'Storage A', 'Lab 1', 'Debris 1', 'Maintenance Corridor', 'Survivor Room']


In [35]:
graph = {
    "Control Room": ["Storage A", "Storage B"],
    "Storage A": ["Lab 1", "Lab 2"],
    "Storage B": ["Server Room"],
    "Lab 1": ["Debris 1"],
    "Lab 2": ["Debris 2"],
    "Debris 1": ["Maintenance Corridor"],
    "Debris 2": ["Maintenance Corridor"],
    "Server Room": ["Charging Bay"],
    "Charging Bay": ["Maintenance Corridor"],
    "Maintenance Corridor": ["Survivor Room"],
    "Survivor Room": []
}

def dfs_iterative(graph, start, goal):
    visited = set()
    stack = [start]
    visited.add(start)
    parent = {start: None}
    expansion_order = []

    while stack:
        node = stack.pop()
        expansion_order.append(node)

        if node == goal:
            path = []
            curr = node
            while curr is not None:
                path.append(curr)
                curr = parent[curr]
            path.reverse()
            return expansion_order, path

        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                visited.add(neighbor)
                parent[neighbor] = node
                stack.append(neighbor)

    return expansion_order, None

start_node = "Control Room"
goal_node = "Survivor Room"
order, path = dfs_iterative(graph, start_node, goal_node)

print("DFS Iterative Order of node expansion:", order)
print("DFS Iterative Path to goal:", path)

DFS Iterative Order of node expansion: ['Control Room', 'Storage B', 'Server Room', 'Charging Bay', 'Maintenance Corridor', 'Survivor Room']
DFS Iterative Path to goal: ['Control Room', 'Storage B', 'Server Room', 'Charging Bay', 'Maintenance Corridor', 'Survivor Room']


In [36]:
graph = {
    "Central Command": [("Shelter A", 4), ("Shelter B", 2)],
    "Shelter A": [("Street-1", 5), ("Street-2", 3)],
    "Shelter B": [("Street-2", 2)],
    "Street-1": [("Hospital Zone", 7)],
    "Street-2": [("Hospital Zone", 4), ("Street-3", 6)],
    "Street-3": [("Hospital Zone", 2)],
    "Hospital Zone": []
}

def ucs(graph, start, goal):
    frontier = [(0, start)]
    best_cost = {start: 0}
    parent = {start: None}
    expansion_order = []

    while frontier:
        min_index = 0
        for i in range(1, len(frontier)):
            if frontier[i][0] < frontier[min_index][0]:
                min_index = i
        cost, node = frontier.pop(min_index)

        if cost > best_cost[node]:
            continue

        expansion_order.append(node)

        if node == goal:
            path = []
            curr = node
            while curr is not None:
                path.append(curr)
                curr = parent[curr]
            path.reverse()
            return expansion_order, path, cost

        for neighbor, edge_cost in graph.get(node, []):
            new_cost = cost + edge_cost
            if neighbor not in best_cost or new_cost < best_cost[neighbor]:
                best_cost[neighbor] = new_cost
                parent[neighbor] = node
                frontier.append((new_cost, neighbor))

    return expansion_order, None, None

start = "Central Command"
goal = "Hospital Zone"
order, path, total_cost = ucs(graph, start, goal)

print("UCS Order of node expansion:", order)
print("UCS Path to goal:", path)
print("Total cost:", total_cost)

UCS Order of node expansion: ['Central Command', 'Shelter B', 'Shelter A', 'Street-2', 'Hospital Zone']
UCS Path to goal: ['Central Command', 'Shelter B', 'Street-2', 'Hospital Zone']
Total cost: 8


In [37]:
graph = {
    "Entrance": [("Room A", 2), ("Room B", 3)],
    "Room A": [("Room C", 2), ("Room D", 5)],
    "Room B": [("Room D", 2)],
    "Room C": [("Treasure Room 1", 4)],
    "Room D": [("Treasure Room 2", 3), ("Trap Room", 1)],
    "Trap Room": [("Treasure Room 3", 2)],
    "Treasure Room 1": [],
    "Treasure Room 2": [],
    "Treasure Room 3": []
}

treasure_values = {
    "Treasure Room 1": 10,
    "Treasure Room 2": 8,
    "Treasure Room 3": 15
}

def ucs_treasure_hunt(graph, start, treasure_values):
    frontier = [(0, start)]
    best_cost = {start: 0}
    parent = {start: None}
    expansion_order = []
    treasure_costs = {}

    while frontier:
        min_index = 0
        for i in range(1, len(frontier)):
            if frontier[i][0] < frontier[min_index][0]:
                min_index = i
        cost, node = frontier.pop(min_index)

        if cost > best_cost[node]:
            continue

        expansion_order.append(node)

        if node in treasure_values:
            treasure_costs[node] = cost

        for neighbor, edge_cost in graph.get(node, []):
            new_cost = cost + edge_cost
            if neighbor not in best_cost or new_cost < best_cost[neighbor]:
                best_cost[neighbor] = new_cost
                parent[neighbor] = node
                frontier.append((new_cost, neighbor))

    best_utility = float('-inf')
    best_treasure = None
    for treasure, cost in treasure_costs.items():
        utility = treasure_values[treasure] - cost
        if utility > best_utility:
            best_utility = utility
            best_treasure = treasure

    path = []
    if best_treasure:
        curr = best_treasure
        while curr is not None:
            path.append(curr)
            curr = parent[curr]
        path.reverse()

    return expansion_order, path, best_utility

def compute_path_cost(graph, path):
    if len(path) < 2:
        return 0
    total = 0
    for i in range(len(path) - 1):
        for neighbor, cost in graph[path[i]]:
            if neighbor == path[i + 1]:
                total += cost
                break
    return total

start = "Entrance"
order, path, utility = ucs_treasure_hunt(graph, start, treasure_values)
cost = compute_path_cost(graph, path)
treasure = path[-1]
value = treasure_values[treasure]

print("Node expansion order:", order)
print("Path to goal:", " -> ".join(path))
print(f"Total utility: {value} - {cost} = {utility}")

Node expansion order: ['Entrance', 'Room A', 'Room B', 'Room C', 'Room D', 'Trap Room', 'Treasure Room 1', 'Treasure Room 2', 'Treasure Room 3']
Path to goal: Entrance -> Room B -> Room D -> Trap Room -> Treasure Room 3
Total utility: 15 - 8 = 7


In [38]:
from collections import deque

grid = [
    ['S', 1, '#', 2, 3],
    [2, '#', 2, '#', 1],
    [3, 2, 1, 2, 2],
    ['#', 2, '#', 1, 3],
    [2, 1, 2, 2, 'G']
]

rows, cols = 5, 5
start = (0, 0)
goal = (4, 4)

# Directions: up, down, left, right
directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def bfs():
    queue = deque([start])
    visited = set([start])
    parent = {start: None}
    expansion_order = []

    while queue:
        r, c = queue.popleft()
        expansion_order.append((r, c))

        if (r, c) == goal:
            # Reconstruct path
            path = []
            cur = goal
            while cur is not None:
                path.append(cur)
                cur = parent[cur]
            path.reverse()
            return expansion_order, path

        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in visited:
                if grid[nr][nc] != '#':  # not an obstacle
                    visited.add((nr, nc))
                    parent[(nr, nc)] = (r, c)
                    queue.append((nr, nc))

    return expansion_order, None

order, path = bfs()
print("BFS Expansion order:", order)
print("BFS Path:", path)

BFS Expansion order: [(0, 0), (1, 0), (0, 1), (2, 0), (2, 1), (3, 1), (2, 2), (4, 1), (1, 2), (2, 3), (4, 0), (4, 2), (3, 3), (2, 4), (4, 3), (3, 4), (1, 4), (4, 4)]
BFS Path: [(0, 0), (1, 0), (2, 0), (2, 1), (3, 1), (4, 1), (4, 2), (4, 3), (4, 4)]


In [39]:
grid = [
    ['S', 1, '#', 2, 3],
    [2, '#', 2, '#', 1],
    [3, 2, 1, 2, 2],
    ['#', 2, '#', 1, 3],
    [2, 1, 2, 2, 'G']
]

rows, cols = 5, 5
start = (0, 0)
goal = (4, 4)

directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def dfs():
    stack = [start]
    visited = set([start])
    parent = {start: None}
    expansion_order = []

    while stack:
        r, c = stack.pop()
        expansion_order.append((r, c))

        if (r, c) == goal:
            path = []
            cur = goal
            while cur is not None:
                path.append(cur)
                cur = parent[cur]
            path.reverse()
            return expansion_order, path

        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in visited:
                if grid[nr][nc] != '#':
                    visited.add((nr, nc))
                    parent[(nr, nc)] = (r, c)
                    stack.append((nr, nc))

    return expansion_order, None

order, path = dfs()
print("DFS Expansion order:", order)
print("DFS Path:", path)

DFS Expansion order: [(0, 0), (0, 1), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4)]
DFS Path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4)]


In [40]:
import heapq

grid = [
    ['S', 1, '#', 2, 3],
    [2, '#', 2, '#', 1],
    [3, 2, 1, 2, 2],
    ['#', 2, '#', 1, 3],
    [2, 1, 2, 2, 'G']
]

rows, cols = 5, 5
start = (0, 0)
goal = (4, 4)

directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def cost_of_cell(r, c):
    """Return the cost to enter this cell (start and goal cost 0)."""
    val = grid[r][c]
    if val == 'S' or val == 'G':
        return 0
    return int(val)

def ucs():
    # priority queue: (total_cost, row, col)
    pq = [(0, start[0], start[1])]
    best_cost = {start: 0}
    parent = {start: None}
    expansion_order = []

    while pq:
        cost, r, c = heapq.heappop(pq)

        if (r, c) in best_cost and cost > best_cost[(r, c)]:
            continue

        expansion_order.append((r, c))

        if (r, c) == goal:
            path = []
            cur = goal
            while cur is not None:
                path.append(cur)
                cur = parent[cur]
            path.reverse()
            return expansion_order, path, cost

        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] != '#':
                new_cost = cost + cost_of_cell(nr, nc)
                if (nr, nc) not in best_cost or new_cost < best_cost[(nr, nc)]:
                    best_cost[(nr, nc)] = new_cost
                    parent[(nr, nc)] = (r, c)
                    heapq.heappush(pq, (new_cost, nr, nc))

    return expansion_order, None, None

order, path, total_cost = ucs()
print("UCS Expansion order:", order)
print("UCS Path:", path)
print("Total cost:", total_cost)

UCS Expansion order: [(0, 0), (0, 1), (1, 0), (2, 0), (2, 1), (2, 2), (3, 1), (1, 2), (2, 3), (4, 1), (3, 3), (2, 4), (4, 0), (4, 2), (1, 4), (4, 3), (4, 4)]
UCS Path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (3, 3), (4, 3), (4, 4)]
Total cost: 13


In [41]:
import heapq

# Grid: rows A-E (0-4), columns 1-5 (0-4)
grid = [
    ['S', 4, '#', 2, 0],
    [3, '#', 3, '#', 1],
    [2, 2, 2, 2, 1],
    ['#', 3, '#', 1, 2],
    [3, 2, 3, 2, 'G']
]

rows, cols = 5, 5
start = (0, 0)
goal = (4, 4)

# Directions: up, down, left, right
directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def heuristic(r, c):
    """Return heuristic value for a cell (goal = 0, start = 0)."""
    if (r, c) == goal:
        return 0
    val = grid[r][c]
    if val == 'S' or val == '#':
        return 0  # start has no heuristic; obstacles are never visited
    return int(val)

def greedy_best_first():
    # Priority queue: (heuristic, row, col)
    pq = [(heuristic(start[0], start[1]), start[0], start[1])]
    visited = set()
    parent = {start: None}
    expansion_order = []

    while pq:
        h, r, c = heapq.heappop(pq)
        if (r, c) in visited:
            continue
        visited.add((r, c))
        expansion_order.append((r, c))

        if (r, c) == goal:
            # Reconstruct path
            path = []
            cur = goal
            while cur is not None:
                path.append(cur)
                cur = parent[cur]
            path.reverse()
            return expansion_order, path

        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols:
                if grid[nr][nc] != '#' and (nr, nc) not in visited:
                    if (nr, nc) not in parent:
                        parent[(nr, nc)] = (r, c)
                    heapq.heappush(pq, (heuristic(nr, nc), nr, nc))

    return expansion_order, None

order, path = greedy_best_first()
print("Greedy Best-First Expansion order:", order)
print("Greedy Best-First Path:", path)

Greedy Best-First Expansion order: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (1, 4), (0, 4), (3, 3), (0, 3), (3, 4), (4, 4)]
Greedy Best-First Path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4)]


In [42]:
import random
N = 7
POPULATION_SIZE = 100
MAX_GENERATIONS = 1000
MUTATION_RATE = 0.1
TOURNAMENT_SIZE = 3
MAX_FITNESS = N * (N - 1) // 2   # 21
# PROCESS 1: Chromosome Representation (Random Permutation)
def create_chromosome():
    return random.sample(range(N), N)
# PROCESS 2: Fitness Function (Count Non-Attacking Pairs)
def fitness(chromosome):
    non_attacking = 0
    for i in range(N):
        for j in range(i + 1, N):
            if abs(chromosome[i] - chromosome[j]) != abs(i - j):
                non_attacking += 1
    return non_attacking
# PROCESS 3: Selection (Tournament Selection)
def tournament_selection(population):
    selected = []
    for _ in range(2):  # select two parents
        tournament = random.sample(population, TOURNAMENT_SIZE)
        best = max(tournament, key=fitness)
        selected.append(best)
    return selected[0], selected[1]
# PROCESS 4: Crossover (Order Crossover - OX)
def order_crossover(parent1, parent2):
    size = len(parent1)
    start, end = sorted(random.sample(range(size), 2))
    child = [None] * size

    child[start:end+1] = parent1[start:end+1]

    current = 0
    for gene in parent2:
        if gene not in child:
            while child[current] is not None:
                current += 1
            child[current] = gene
    return child
# PROCESS 5: Mutation (Swap Mutation)
def swap_mutation(chromosome):
    idx1, idx2 = random.sample(range(N), 2)
    chromosome[idx1], chromosome[idx2] = chromosome[idx2], chromosome[idx1]
    return chromosome
# PROCESS 6: TERMINATION CONDITION & Genetic Algorithm Main Loop 
def genetic_algorithm():

    population = [create_chromosome() for _ in range(POPULATION_SIZE)]
    best_solution = None
    best_fitness = -1

    for generation in range(MAX_GENERATIONS):

        fitness_scores = [fitness(ind) for ind in population]
        current_best = max(fitness_scores)
        if current_best > best_fitness:
            best_fitness = current_best
            best_solution = population[fitness_scores.index(current_best)]

        # TERMINATION CONDITION: Check if we found a perfect solution
        if best_fitness == MAX_FITNESS:
            print(f"Perfect solution found at generation {generation+1}")
            break

        new_population = []
        while len(new_population) < POPULATION_SIZE:
            parent1, parent2 = tournament_selection(population)
            child = order_crossover(parent1, parent2)
            if random.random() < MUTATION_RATE:
                child = swap_mutation(child)
            new_population.append(child)

        population = new_population

    return best_solution, best_fitness
# SOLUTION: Run GA and Display Result
solution, fit = genetic_algorithm()
print(f"Camera Placement: {solution}")
print(f"Fitness: {fit}")

Perfect solution found at generation 1
Camera Placement: [2, 6, 3, 0, 4, 1, 5]
Fitness: 21


In [43]:
import random

chromosomes = ['C1', 'C2', 'C3', 'C4', 'C5']
fitnesses = [12, 20, 15, 8, 25]
# PROCESS 1: Roulette Wheel Selection
# DESCRIPTION:
# Each individual gets a slice of a roulette wheel proportional to its fitness.
# A random number selects the winner.
def roulette_wheel_selection(chromosomes, fitnesses):
    total = sum(fitnesses)
    pick = random.uniform(0, total)
    cumulative = 0
    for ch, fit in zip(chromosomes, fitnesses):
        cumulative += fit
        if cumulative > pick:
            return ch
    return chromosomes[-1]
# PROCESS 2: Tournament Selection
# DESCRIPTION:
# Randomly pick k individuals, then return the one with the highest fitness.
def tournament_selection(chromosomes, fitnesses, k=3):
    participants = random.sample(list(zip(chromosomes, fitnesses)), k)
    winner = max(participants, key=lambda x: x[1])[0]
    return winner
# PROCESS 3: Rank Selection
# DESCRIPTION:
# Sort by fitness, assign ranks (1 = worst, N = best).
# Probability is proportional to rank (higher rank → higher chance).
def rank_selection(chromosomes, fitnesses):

    paired = sorted(zip(chromosomes, fitnesses), key=lambda x: x[1])
    n = len(paired)
    ranks = list(range(1, n+1))
    total_rank = sum(ranks)
    probs = [r / total_rank for r in ranks]

    pick = random.uniform(0, 1)
    cumulative = 0
    for (ch, _), p in zip(paired, probs):
        cumulative += p
        if cumulative > pick:
            return ch
    return paired[-1][0]
# LAST STEP: Test each selection technique multiple times
print("=== Roulette Wheel Selection (5 runs) ===")
for _ in range(5):
    print(roulette_wheel_selection(chromosomes, fitnesses))

print("\n=== Tournament Selection (5 runs) ===")
for _ in range(5):
    print(tournament_selection(chromosomes, fitnesses))

print("\n=== Rank Selection (5 runs) ===")
for _ in range(5):
    print(rank_selection(chromosomes, fitnesses))

=== Roulette Wheel Selection (5 runs) ===
C2
C2
C5
C5
C3

=== Tournament Selection (5 runs) ===
C2
C5
C5
C2
C2

=== Rank Selection (5 runs) ===
C2
C2
C5
C3
C2


In [44]:
import random

parent1 = [1, 3, 5, 0, 6, 4, 2]
parent2 = [2, 5, 1, 6, 0, 3, 4]
# PROCESS 1: One-Point Crossover (with duplicate repair)
def one_point_crossover(p1, p2):
    n = len(p1)
    point = random.randint(1, n - 1)
    child = p1[:point] + p2[point:]

    seen = set()
    duplicates = []
    for i, val in enumerate(child):
        if val in seen:
            duplicates.append(i)
        else:
            seen.add(val)
    missing = [x for x in range(n) if x not in seen]
    for idx, new_val in zip(duplicates, missing):
        child[idx] = new_val
    return child
# PROCESS 2: Two-Point Crossover (with duplicate repair)
def two_point_crossover(p1, p2):
    n = len(p1)
    points = sorted(random.sample(range(1, n), 2))
    child = p1[:points[0]] + p2[points[0]:points[1]] + p1[points[1]:]

    seen = set()
    duplicates = []
    for i, val in enumerate(child):
        if val in seen:
            duplicates.append(i)
        else:
            seen.add(val)
    missing = [x for x in range(n) if x not in seen]
    for idx, new_val in zip(duplicates, missing):
        child[idx] = new_val
    return child
# PROCESS 3: Order Crossover (OX)
def order_crossover(p1, p2):
    n = len(p1)
    start, end = sorted(random.sample(range(n), 2))
    child = [None] * n

    child[start:end+1] = p1[start:end+1]

    current = 0
    for gene in p2:
        if gene not in child:
            while child[current] is not None:
                current += 1
            child[current] = gene
    return child
# LAST STEP: Test each crossover technique
print("One-Point Crossover:", one_point_crossover(parent1, parent2))
print("Two-Point Crossover:", two_point_crossover(parent1, parent2))
print("Order Crossover:", order_crossover(parent1, parent2))

One-Point Crossover: [1, 3, 5, 0, 2, 6, 4]
Two-Point Crossover: [1, 3, 5, 6, 0, 4, 2]
Order Crossover: [2, 1, 5, 0, 6, 4, 3]


In [45]:
import random

chromosome = [1, 3, 5, 0, 6, 4, 2]
# PROCESS 1: Swap Mutation
def swap_mutation(chromo):
    idx1, idx2 = random.sample(range(len(chromo)), 2)
    chromo[idx1], chromo[idx2] = chromo[idx2], chromo[idx1]
    return chromo
# PROCESS 2: Scramble Mutation
def scramble_mutation(chromo):
    start = random.randint(0, len(chromo) - 1)
    end = random.randint(start, len(chromo) - 1)
    segment = chromo[start:end+1]
    random.shuffle(segment)
    chromo[start:end+1] = segment
    return chromo
# PROCESS 3: Inversion Mutation
def inversion_mutation(chromo):
    start = random.randint(0, len(chromo) - 1)
    end = random.randint(start, len(chromo) - 1)
    chromo[start:end+1] = reversed(chromo[start:end+1])
    return chromo
# LAST STEP: Test each mutation technique
print("Swap Mutation:", swap_mutation(chromosome.copy()))
print("Scramble Mutation:", scramble_mutation(chromosome.copy()))
print("Inversion Mutation:", inversion_mutation(chromosome.copy()))

Swap Mutation: [1, 3, 5, 6, 0, 4, 2]
Scramble Mutation: [1, 3, 5, 0, 6, 4, 2]
Inversion Mutation: [1, 3, 6, 0, 5, 4, 2]
